# Creating dataset for training YOLO model with RTMO-X

### Dataset construction with RTMO-X

In [1]:
# DEPENDENCIES

import glob
import cv2
import torch

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(device)
print(torch.__version__)

from pathlib import Path
from mmpose.apis import init_model, inference_topdown
from mmpose.visualization import PoseLocalVisualizer
from mmdet.apis import init_detector, inference_detector
from enum import Enum
from typing import List

cuda
2.1.2+cu118


d:\ProgramFiles\Anaconda\envs\openmmlab-gpu\lib\site-packages\albumentations\__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [2]:
# SETUP

KEYPOINT_IDS = [
    "nose",
    "left_eye",
    "right_eye",
    "left_ear",
    "right_ear",
    "left_shoulder",
    "right_shoulder",
    "left_elbow",
    "right_elbow",
    "left_wrist",
    "right_wrist",
    "left_hip",
    "right_hip",
    "left_knee",
    "right_knee",
    "left_ankle",
    "right_ankle",
]

CONFIG_ROOT_PATH = "D:\\Magistrska\\mmpose\\configs\\"
CHECKPOINT_ROOT_PATH = "D:\\Magistrska\\mmpose\\checkpoints\\"


def get_model_cfg_and_checkpoint():
    cfg_path = f"{CONFIG_ROOT_PATH}body_2d_keypoint\\rtmo\\coco\\rtmo-x_8xb256-700e_coco-384x288.py"
    checkpoint_path = f"{CHECKPOINT_ROOT_PATH}rtmo\\coco\\rtmpose-x_simcc-body7_pt-body7_700e-384x288-71d7b7e9_20230629.pth"

    return cfg_path, checkpoint_path


def get_detector_cfg_and_checkpoint():
    cfg_path = f"{CONFIG_ROOT_PATH}body_2d_keypoint\\yoloxpose\\coco\\yoloxpose_l_8xb32-300e_coco-640.py"
    checkpoint_path = f"{CHECKPOINT_ROOT_PATH}yolox\\yoloxpose_l_8xb32-300e_coco-640-de0f8dee_20230829.pth"

    return cfg_path, checkpoint_path

In [3]:
# KEYPOINT UTIL


class Keypoint:
    def __init__(self, id, x, y):
        self.id = id
        self.pixelPosition = (x, y)


def processKeypoints(keypoints: list, width: int, height: int) -> List[Keypoint]:
    final_keypoints: list[Keypoint] = []

    for i, keypoint in enumerate(keypoints):
        keypoint_id = KEYPOINT_IDS[i]
        x = float(keypoint[0]) / width
        y = float(keypoint[1]) / height
        final_keypoints.append(Keypoint(keypoint_id, x, y))

    return final_keypoints

def createTxtFileFromKeypoints(img_name: str, width: int, height: int, keypoints: List[Keypoint], output_dir: str): #
    txt_filename = f"{Path(img_name).stem}.txt"
    output_path = Path(output_dir) / txt_filename

    person_class = 0

    x_min = min(k.pixelPosition[0] for k in keypoints)
    x_max = max(k.pixelPosition[0] for k in keypoints)
    y_min = min(k.pixelPosition[1] for k in keypoints)
    y_max = max(k.pixelPosition[1] for k in keypoints)

    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2

    bbox_width = x_max - x_min
    bbox_height = y_max - y_min

    keypoints_string = ''.join([f"{kp.pixelPosition[0]:.6f} {kp.pixelPosition[1]:.6f} " for kp in keypoints]).strip()

    out_string = f"{person_class} {x_center:.6f} {y_center:.6f} {bbox_width} {bbox_height} {keypoints_string}"

    with open(output_path, 'w') as f:
        f.write(out_string)

In [4]:
# CONSTANTS

DATASET_DIR = r'D:\Magistrska\blindoff-dataset\images'
IMG_EXTS = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp", "*.tif", "*.tiff")

In [ ]:
# RECOGNITION ON IMAGES

model_cfg_path, model_checkpoint_path = get_model_cfg_and_checkpoint()
detector_cfg_path, detector_checkpoint_path = get_detector_cfg_and_checkpoint()

pose_model = init_model(model_cfg_path, model_checkpoint_path, device=device)
det_model = init_detector(detector_cfg_path, detector_checkpoint_path, device=device)

input_dir = Path(DATASET_DIR)

image_paths = []
for ext in IMG_EXTS:
    image_paths.extend(glob.glob(str(input_dir / ext)))
image_paths = sorted(image_paths)

total = len(image_paths)

for idx, img_path in enumerate(image_paths, start=1):
    print(f"Processing image ({idx}/{total})")

    frame = cv2.imread(img_path)
    if frame is None:
        print(f"[WARN] Could not read image: {img_path}")
        continue

    # --------- BBoxes ---------
    det_result = inference_detector(det_model, frame)
    pred_instance = det_result.pred_instances
    person_bboxes = pred_instance.bboxes[pred_instance.labels == 0]
    scores = pred_instance.scores[pred_instance.labels == 0]

    if len(person_bboxes) > 0:
        best_idx = scores.argmax().item()
        bboxes = person_bboxes[best_idx].reshape(1, -1)
    else:
        bboxes = []

    # --------- Pose inference ---------
    model_pose_results = inference_topdown(pose_model, frame, bboxes)

    # --------- Visualization ---------
    vis_frame = frame.copy()
    pose = model_pose_results[0]

    w, h = frame.shape[1], frame.shape[0]
    keypoints = processKeypoints(
        pose.pred_instances.keypoints[0], w, h 
    )

    createTxtFileFromKeypoints(img_path, w, h, keypoints, output_dir=input_dir)

Loads checkpoint by local backend from path: D:\Magistrska\mmpose\checkpoints\rtmo\coco\rtmpose-x_simcc-body7_pt-body7_700e-384x288-71d7b7e9_20230629.pth


d:\magistrska\mmpose\mmpose\datasets\datasets\utils.py:102: UserWarning: The metainfo config file "configs/_base_/datasets/coco.py" does not exist. A matched config file "d:\magistrska\mmpose\mmpose\.mim\configs\_base_\datasets\coco.py" will be used instead.
  warnings.warn(


Loads checkpoint by local backend from path: D:\Magistrska\mmpose\checkpoints\yolox\yoloxpose_l_8xb32-300e_coco-640-de0f8dee_20230829.pth


d:\ProgramFiles\Anaconda\envs\openmmlab-gpu\lib\site-packages\mmdet\apis\inference.py:108: UserWarning: palette does not exist, random is used by default. You can also set the palette to customize.
  warnings.warn(


Processing image (3732/6295)


d:\ProgramFiles\Anaconda\envs\openmmlab-gpu\lib\site-packages\torch\functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\TensorShape.cpp:3527.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Processing image (3733/6295)
Processing image (3734/6295)
Processing image (3735/6295)
Processing image (3736/6295)
Processing image (3737/6295)
Processing image (3738/6295)
Processing image (3739/6295)
Processing image (3740/6295)
Processing image (3741/6295)
Processing image (3742/6295)
Processing image (3743/6295)
Processing image (3744/6295)
Processing image (3745/6295)
Processing image (3746/6295)
Processing image (3747/6295)
Processing image (3748/6295)
Processing image (3749/6295)
Processing image (3750/6295)
Processing image (3751/6295)
Processing image (3752/6295)
Processing image (3753/6295)
Processing image (3754/6295)
Processing image (3755/6295)
Processing image (3756/6295)
Processing image (3757/6295)
Processing image (3758/6295)
Processing image (3759/6295)
Processing image (3760/6295)
Processing image (3761/6295)
Processing image (3762/6295)
Processing image (3763/6295)
Processing image (3764/6295)
Processing image (3765/6295)
Processing image (3766/6295)
Processing ima

### Dataset Spliting

In [7]:
from pathlib import Path
import random
import shutil

# =========================
# CONFIG
# =========================
DATASET_ROOT = Path(r"D:\Magistrska\blindoff-dataset")     # change if needed
SRC_IMAGES_DIR = DATASET_ROOT / "images"    # your current flat folder with images + txt
OUT_IMAGES_DIR = DATASET_ROOT / "dataset/images"
OUT_LABELS_DIR = DATASET_ROOT / "dataset/labels"

TRAIN_RATIO = 0.8
VAL_RATIO   = 0.1
TEST_RATIO  = 0.1

SEED = 42
MOVE_FILES = False   # False = copy, True = move

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# =========================
# HELPERS
# =========================
def ensure_dirs():
    for split in ("train", "val", "test"):
        (OUT_IMAGES_DIR / split).mkdir(parents=True, exist_ok=True)
        (OUT_LABELS_DIR / split).mkdir(parents=True, exist_ok=True)

def split_pairs(pairs, train_ratio, val_ratio, seed=42):
    random.seed(seed)
    random.shuffle(pairs)
    n = len(pairs)
    n_train = int(n * train_ratio)
    n_val   = int(n * val_ratio)
    # remainder -> test
    train = pairs[:n_train]
    val   = pairs[n_train:n_train + n_val]
    test  = pairs[n_train + n_val:]
    return train, val, test

def op_func(move: bool):
    return shutil.move if move else shutil.copy2

# =========================
# 1) COLLECT (image, label) PAIRS
# =========================
pairs = []
missing_labels = []
for img_path in SRC_IMAGES_DIR.iterdir():
    if img_path.is_file() and img_path.suffix.lower() in IMG_EXTS:
        label_path = img_path.with_suffix(".txt")
        if label_path.exists():
            pairs.append((img_path, label_path))
        else:
            missing_labels.append(img_path.name)

print(f"Found {len(pairs)} image+label pairs.")
if missing_labels:
    print(f"WARNING: {len(missing_labels)} images are missing .txt labels (showing up to 10):")
    print(missing_labels[:10])

if len(pairs) == 0:
    raise RuntimeError("No valid (image, .txt) pairs found. Check SRC_IMAGES_DIR and extensions.")

# =========================
# 2) SPLIT
# =========================
if abs((TRAIN_RATIO + VAL_RATIO + TEST_RATIO) - 1.0) > 1e-9:
    raise ValueError("TRAIN_RATIO + VAL_RATIO + TEST_RATIO must sum to 1.0")

train_pairs, val_pairs, test_pairs = split_pairs(pairs, TRAIN_RATIO, VAL_RATIO, seed=SEED)

print(f"Split counts -> train: {len(train_pairs)}, val: {len(val_pairs)}, test: {len(test_pairs)}")

# =========================
# 3) CREATE OUTPUT DIRS
# =========================
ensure_dirs()

# =========================
# 4) COPY/MOVE FILES
# =========================
op = op_func(MOVE_FILES)

def place(split_name, items):
    for img_path, label_path in items:
        # image -> images/split
        op(str(img_path), str(OUT_IMAGES_DIR / split_name / img_path.name))
        # label -> labels/split
        op(str(label_path), str(OUT_LABELS_DIR / split_name / label_path.name))

place("train", train_pairs)
place("val",   val_pairs)
place("test",  test_pairs)

print("Done.")
print("Output structure created under:", DATASET_ROOT.resolve())

# =========================
# OPTIONAL: if you want to remove leftover stray .txts in SRC_IMAGES_DIR
# (only do this if MOVE_FILES=True or if you know what you're doing)
# =========================
# stray_txts = [p for p in SRC_IMAGES_DIR.glob("*.txt")]
# print(f"Stray txt files still in {SRC_IMAGES_DIR}: {len(stray_txts)}")


Found 9429 image+label pairs.
['TBsYULM5BWehuDzLW4NF_fwHXHa8KwENetE8ZvnWvxQ25aFx1_carioca-squat_1_5.jpg']
Split counts -> train: 7543, val: 942, test: 944
Done.
Output structure created under: D:\Magistrska\blindoff-dataset
